# Preprocessing Practice: Your Turn

Now you clean a dataset yourself. This notebook has the same **structure** as `04_preprocessing_demo.ipynb`, but the code cells are empty (or partially started) -- you write the cleaning steps.

**The dataset:** `data/broken_dataset_practice.csv` -- online delivery orders (`delivery_days`: how many days a delivery took). It is a *different* dataset from the demo, with different column names and different specific problems, but the same six categories are hiding in it (Part II, slide 26):

- Missing values
- Duplicate records
- Outliers
- Wrong data types
- Class imbalance / skew (this one is a regression target, so watch for a *skewed* distribution rather than imbalanced classes)
- Data leakage

**How to use this notebook:**
1. Work through the sections in order -- each one tells you *what* to check, not exactly *how* (that's the part you practice).
2. If you get stuck, the pattern from notebook 04 applies directly -- go back and look at how that problem was solved there.
3. Every section ends with a `# TODO` cell for you to fill in, and most also have a hint you can peek at if needed.
4. The final cell is a self-check -- if all the numbers come back clean, you're done.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (7, 4.5)

print("Environment ready.")

## Step 1: Load and Take a First Look

Load the CSV, then look before assuming anything (Part II, slide 24).

In [ ]:
df = pd.read_csv("data/broken_dataset_practice.csv")
print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)

In [ ]:
# TODO: run df.info() and df.describe() here.
# Look for anything that doesn't match what the column name promises
# (e.g. a numeric-sounding column that isn't stored as a number).


## Step 2: Missing Values

Find how much is missing per column, and per-column percentage. Remember the rule from notebook 04: a small gap is safe to impute, but past roughly half a column missing, dropping the column is usually the better call.

In [ ]:
# TODO: compute missing count and missing percentage per column.
# Hint: df.isna().sum(), and divide by len(df) * 100 for the percentage.


In [ ]:
# TODO: for the column with a LOT missing (>50%), drop it.
# TODO: for the column with a small amount missing, fill it with the median (numeric)
#       or the most common value / mode (categorical).
#
# df = df.drop(columns=[...])
# df["..."] = df["..."].fillna(df["..."].median())


## Step 3: Duplicate Records

Check for exact duplicate rows and remove them now, before any splitting or further cleaning.

In [ ]:
# TODO: count duplicate rows, print the count, then drop them.
# Hint: df.duplicated().sum(), then df.drop_duplicates().reset_index(drop=True)


## Step 4: Wrong Data Types

There are **two** dtype problems in this dataset -- notebook 04's demo only had one, so look carefully:

1. A column that should be numeric but has text mixed in (same pattern as `hours_studied` in the demo -- some values might have a unit suffix attached).
2. A column that holds dates but is stored as plain text, so pandas can't do date math on it.

Use `df[col].sample(10)` on any column you're unsure about to see what's actually inside it, the same way we did in notebook 04.

In [ ]:
# TODO: inspect suspect columns with df[col].sample(10) to see their raw values.


In [ ]:
# TODO: fix the numeric-but-text column.
# Hint (same pattern as notebook 04):
# df["..."] = df["..."].astype(str).str.replace("kg", "", regex=False).astype(float)

# TODO: fix the date-but-text column.
# Hint: df["..."] = pd.to_datetime(df["..."], format="mixed")
# (or check the exact format printed above and pass format="%d/%m/%Y" etc.)

# After both fixes, re-check df.dtypes to confirm.


**Remember the lesson from notebook 04:** fixing a dtype can reveal duplicates that inconsistent text formatting was hiding. Worth a re-check before moving on.

In [ ]:
# TODO: check df.duplicated().sum() again. If it's > 0, drop them.


# Leave this line as-is -- it snapshots the duplicate count for the final check below.
# (Checking again after Step 8 wouldn't be meaningful: once order_id is dropped for
# modelling, two genuinely different orders can coincidentally share every remaining
# feature value without being data-quality duplicates.)
duplicates_handled = df.duplicated().sum() == 0
print(f"Duplicates handled: {duplicates_handled}")

## Step 5: Outliers

One numeric column has a handful of impossible values (think about which real-world quantity in this dataset has a natural upper limit, then look for values far past it). Use a box plot to see it, then the IQR rule to find and remove the exact rows.

In [ ]:
# TODO: box plot the numeric columns to spot which one has outliers.
# Hint: df.boxplot(column="...") for one column, or df[numeric_cols].boxplot() for several.


In [ ]:
# TODO: use the IQR rule to find and drop the outlier rows.
# Hint (same pattern as notebook 04):
# q1, q3 = df["..."].quantile([0.25, 0.75])
# iqr = q3 - q1
# lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
# outlier_mask = (df["..."] < lower) | (df["..."] > upper)
# df = df.loc[~outlier_mask].reset_index(drop=True)


## Step 6: Data Leakage

The target here is `delivery_days`. One column in this dataset only makes sense *after* a delivery has already happened -- ask yourself, for each remaining column, "would I know this value at the moment I need to predict delivery time?" Find it, then drop it.

In [ ]:
# TODO: look at the remaining column names. df.columns


In [ ]:
# TODO: drop the leakage column.
# df = df.drop(columns=["..."])


## Step 7: Target Distribution

`delivery_days` is a continuous number (regression), not a category -- so "class imbalance" doesn't directly apply here (Part I, slide 14). The equivalent thing to check for a regression target is **skew**: is the distribution roughly symmetric, or bunched up with a long tail? A skewed target can make some metrics (like RMSE) more sensitive to the tail values than you'd expect.

Plot a histogram of `delivery_days` and look at its shape. Nothing to "fix" necessarily -- just note it, the same way we noted class imbalance in notebook 04.

In [ ]:
# TODO: plot a histogram of delivery_days.
# Hint: df["delivery_days"].hist(bins=30)


## Step 8: Feature Engineering

There's a categorical column (or two) left that needs encoding before this data is model-ready. Find it/them and one-hot encode.

In [ ]:
# TODO: find remaining categorical (text) columns and one-hot encode them.
# Hint: df.select_dtypes(include=["object"]).columns
# then: df = pd.get_dummies(df, columns=[...], drop_first=True)


## Final Check: Self-Grading Cell

Run this last. It doesn't tell you *how* to fix anything -- just whether each problem is actually gone. If something still says FAIL, go back to that section.

In [ ]:
checks = []

checks.append(("No missing values", df.isna().sum().sum() == 0, f"{df.isna().sum().sum()} missing values remain"))

# Uses the snapshot from Step 4, not a fresh check -- see the note there for why.
checks.append(("Duplicates were handled", "duplicates_handled" in dir() and duplicates_handled,
               "see Step 3/4" if not ("duplicates_handled" in dir() and duplicates_handled) else "confirmed"))

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
checks.append(("At least 2 numeric columns", len(numeric_cols) >= 2, f"{len(numeric_cols)} numeric columns found"))

has_leakage_col = "actual_delivery_date_note" in df.columns
checks.append(("No leakage column present", not has_leakage_col, "actual_delivery_date_note still in df" if has_leakage_col else "not present"))

has_text_cols = len(df.select_dtypes(include="object").columns) > 0
checks.append(("No text/categorical columns remain unencoded", not has_text_cols,
               f"still text: {list(df.select_dtypes(include='object').columns)}" if has_text_cols else "all encoded"))

# item_weight_kg might still be text at this point if Step 4 isn't done yet --
# that's a FAIL, not a crash, so guard the numeric comparison.
try:
    extreme_weight = ("item_weight_kg" in df.columns) and (df["item_weight_kg"].astype(float) > 100).any()
    weight_detail = "some values still look like outliers" if extreme_weight else "range looks plausible"
except (ValueError, TypeError):
    extreme_weight = True
    weight_detail = "item_weight_kg is still text -- fix its dtype in Step 4 first"
checks.append(("No impossible item_weight_kg values", not extreme_weight, weight_detail))

print(f"{'CHECK':45s} {'RESULT':8s} DETAIL")
print("-" * 90)
for name, ok, detail in checks:
    print(f"{name:45s} {'PASS' if ok else 'FAIL':8s} {detail}")

passed = sum(1 for _, ok, _ in checks if ok)
print("-" * 90)
print(f"Score: {passed}/{len(checks)}")
if passed == len(checks):
    print("\nAll checks passed -- this dataset is ready for feature engineering / modelling")
    print("(see 03_modelling_template.ipynb for the next step).")
else:
    print("\nSome checks failed -- go back to the matching section above and try again.")